# Week 3 · Course 2 · Linear Regression

**Sections**
1. Simple Linear Regression — model, RSS, least-squares `(0:00 – 0:25)`
2. Assessing Coefficient Accuracy — SE, CI, t-stat, p-value `(0:25 – 0:50)`
3. Model Accuracy — RSE and R² `(0:50 – 1:05)`
4. Multiple Linear Regression — MLR, interpretation, F-statistic `(1:05 – 1:30)`
5. Variable Selection — forward/backward, AIC, BIC `(1:30 – 1:50)`
6. Qualitative Predictors — dummy variables, multi-level factors `(1:50 – 2:10)`
7. Interactions & Non-linearity — interaction terms, polynomial regression `(2:10 – 2:30)`

**Libraries introduced:** `sklearn`, `statsmodels`, `scipy.stats`, `numpy`, `pandas`, `matplotlib`, `seaborn`

## Setup

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# scikit-learn
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.feature_selection import RFE

# statsmodels — for inference (SE, CI, p-values, F-test)
import statsmodels.api as sm
import statsmodels.formula.api as smf

# scipy — for t-distribution and F-distribution
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

### Load the Advertising dataset

200 markets. For each: TV, radio, newspaper budget (\$thousands) and sales (thousands of units).

In [ ]:
# The Advertising dataset is available from the ISLR2 package or we load it directly.
ad_url = "https://www.statlearning.com/s/Advertising.csv"
try:
    adv = pd.read_csv(ad_url, index_col=0)
except Exception:
    # Fallback: reconstruct from ISLR summary statistics (reproducible seed)
    rng = np.random.default_rng(42)
    TV = rng.uniform(0.7, 296.4, 200)
    radio = rng.uniform(0, 49.6, 200)
    newspaper = rng.uniform(0.3, 114.0, 200)
    sales = 2.939 + 0.046*TV + 0.189*radio - 0.001*newspaper + rng.normal(0, 1.69, 200)
    adv = pd.DataFrame({'TV': TV, 'radio': radio, 'newspaper': newspaper, 'sales': sales})

print(adv.shape)
adv.head()

---
## 1. Simple Linear Regression

**Model:** $Y = \beta_0 + \beta_1 X + \varepsilon$

We start with one predictor: **TV → sales**.

### 1.1 Scatter plot — TV vs Sales

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(adv['TV'], adv['sales'], alpha=0.5, edgecolors='k', linewidths=0.4)
ax.set_xlabel('TV budget ($thousands)')
ax.set_ylabel('Sales (thousands of units)')
ax.set_title('Advertising data — TV vs Sales')
plt.tight_layout()
plt.show()

### 1.2 Least-squares estimation — by hand

$$\hat{\beta}_1 = \frac{\sum_{i=1}^n (x_i - \bar{x})(y_i - \bar{y})}{\sum_{i=1}^n (x_i - \bar{x})^2}, \quad \hat{\beta}_0 = \bar{y} - \hat{\beta}_1 \bar{x}$$

In [ ]:
x = adv['TV'].values
y = adv['sales'].values

x_bar, y_bar = x.mean(), y.mean()

beta1_hat = np.sum((x - x_bar) * (y - y_bar)) / np.sum((x - x_bar) ** 2)
beta0_hat = y_bar - beta1_hat * x_bar

print(f"β̂₀ (intercept) = {beta0_hat:.4f}")
print(f"β̂₁ (slope)     = {beta1_hat:.4f}")
print()
print("Interpretation:")
print(f"  Expected sales when TV = 0  → {beta0_hat:.2f}k units")
print(f"  Each extra $1k on TV        → +{beta1_hat*1000:.1f} units sold")

### 1.3 Fitting with scikit-learn

In [ ]:
X_tv = adv[['TV']]   # 2-D array required by sklearn

sk_model = LinearRegression()
sk_model.fit(X_tv, y)

print(f"sklearn intercept : {sk_model.intercept_:.4f}")
print(f"sklearn slope     : {sk_model.coef_[0]:.4f}")

# Residual plot
y_hat = sk_model.predict(X_tv)
residuals = y - y_hat

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Fitted line
tv_range = np.linspace(x.min(), x.max(), 200)
axes[0].scatter(x, y, alpha=0.4, edgecolors='k', linewidths=0.3, label='data')
axes[0].plot(tv_range, sk_model.intercept_ + sk_model.coef_[0]*tv_range,
             color='steelblue', lw=2, label='LS fit')
# Residual segments
for xi, yi, yhi in zip(x, y, y_hat):
    axes[0].plot([xi, xi], [yi, yhi], color='grey', lw=0.5, alpha=0.5)
axes[0].set_xlabel('TV'); axes[0].set_ylabel('Sales')
axes[0].set_title('Fitted line with residuals'); axes[0].legend()

# Residuals vs fitted
axes[1].scatter(y_hat, residuals, alpha=0.4, edgecolors='k', linewidths=0.3)
axes[1].axhline(0, color='red', lw=1, ls='--')
axes[1].set_xlabel('Fitted values'); axes[1].set_ylabel('Residuals')
axes[1].set_title('Residuals vs Fitted')

plt.tight_layout()
plt.show()

### 1.4 Fitting with statsmodels — introduces inference

`statsmodels.OLS` gives us everything sklearn doesn't: standard errors, t-statistics, p-values, confidence intervals, F-statistic.

In [ ]:
X_tv_const = sm.add_constant(X_tv)   # add intercept column
sm_slr = sm.OLS(y, X_tv_const).fit()
print(sm_slr.summary())

**Reading the summary table:**
- `coef` → β̂ values (match our hand calculation)
- `std err` → SE(β̂)
- `t` → t-statistic = coef / std err
- `P>|t|` → two-sided p-value
- `[0.025  0.975]` → 95 % confidence interval

Expected: β̂₀ ≈ 7.03, β̂₁ ≈ 0.0475, t(TV) ≈ 17.67, p < 0.001

---
## 2. Assessing Coefficient Accuracy

### 2.1 Standard errors and 95 % confidence intervals

### Concept: Confidence Interval (CI)

#### What it is
A **95 % confidence interval** is a range of plausible values for a population parameter (e.g., a regression slope β₁).  
If we repeated the experiment 100 times and computed a CI each time, **~95 of those intervals would contain the true β₁**.

> ⚠️ Common misconception: "there is a 95 % chance β₁ is in *this* interval."  
> The true β₁ is fixed — it's either in the interval or it isn't.  
> The 95 % refers to the *procedure*, not one specific interval.

#### Analogy
Imagine you're fishing with a net of fixed width.  
- The **fish** (true β₁) doesn't move.  
- You cast the net at a random spot (your sample).  
- A 95 % CI means: your net-casting method catches the fish 95 % of the time.

#### Formula

$$\hat{\beta}_1 \pm t^*_{\alpha/2,\, n-2} \cdot \text{SE}(\hat{\beta}_1)$$

where:
- $\hat{\beta}_1$ = estimated slope
- $\text{SE}(\hat{\beta}_1) = \hat{\sigma} \,/\, \sqrt{\sum(x_i - \bar{x})^2}$ — measures estimation uncertainty
- $t^*_{\alpha/2,\, n-2}$ = critical value from the t-distribution (≈ 1.96 for large n, α = 0.05)

#### Distribution behind it
The sampling distribution of $\hat{\beta}_1$ follows a **t-distribution** with $n-2$ degrees of freedom:

$$\frac{\hat{\beta}_1 - \beta_1}{\text{SE}(\hat{\beta}_1)} \sim t_{n-2}$$

The CI is simply the interval that captures the central 95 % of this distribution.

In [ ]:
# Standard errors
print("Standard errors:")
print(sm_slr.bse)

# 95 % CI
print("\n95 % confidence intervals:")
ci = sm_slr.conf_int(alpha=0.05)
ci.columns = ['lower', 'upper']
print(ci)

# Visualise the CI for β₁
b1 = sm_slr.params['TV']
lo, hi = ci.loc['TV']
fig, ax = plt.subplots(figsize=(5, 2))
ax.errorbar(b1, 0, xerr=[[b1-lo], [hi-b1]], fmt='o', color='steelblue',
            capsize=8, ms=8, lw=2)
ax.axvline(0, color='red', ls='--', lw=1, label='zero')
ax.set_yticks([])
ax.set_xlabel('β̂₁ (slope for TV)')
ax.set_title('95 % CI for slope — excludes zero → significant')
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nCI: [{lo:.4f}, {hi:.4f}]")

### Concept: Student's t-Test

#### What it is
The **t-test** asks: *"Is this coefficient (or difference) large enough relative to its uncertainty to be considered real?"*

In regression, it tests:
$$H_0: \beta_j = 0 \quad \text{(predictor has no effect)} \qquad \text{vs} \qquad H_a: \beta_j \neq 0$$

#### Analogy
Think of the t-statistic as a **signal-to-noise ratio**:

$$t = \frac{\text{signal (estimated effect)}}{\text{noise (standard error)}} = \frac{\hat{\beta}_j}{\text{SE}(\hat{\beta}_j)}$$

- A large t (|t| > 2) means the signal dominates the noise → likely real effect.
- A small t (|t| < 1) means the effect could easily be noise → no strong evidence.

Just like trying to hear someone across a noisy room: you need a loud-enough voice (large β̂) relative to the background noise (SE).

#### Formula

$$t = \frac{\hat{\beta}_j - 0}{\text{SE}(\hat{\beta}_j)}, \qquad \text{SE}(\hat{\beta}_j) = \hat{\sigma}\sqrt{(X^\top X)^{-1}_{jj}}$$

where $\hat{\sigma}^2 = \text{RSS}/(n-p-1)$ is the estimated error variance.

#### Distribution
Under $H_0$, the test statistic follows a **t-distribution**:

$$t \sim t_{n-p-1}$$

| Degrees of freedom | Shape |
|---|---|
| Small (< 30) | Heavy tails — harder to achieve significance |
| Large (> 100) | Approaches standard Normal $\mathcal{N}(0,1)$ |

The t-distribution's heavy tails account for the extra uncertainty of estimating σ from the data (unlike the z-test which assumes σ is known).

### 2.2 t-statistic and p-value — manually with `scipy.stats`

In [ ]:
# ── Chi-squared test: worked example with the Credit dataset ─────────────────
# Q: Is student status independent of gender?

contingency = pd.crosstab(credit['student'], credit['gender'])
print("Observed counts:")
print(contingency)

chi2_stat, p_val, dof, expected = sp_stats.chi2_contingency(contingency)

print(f"\nExpected counts:")
print(pd.DataFrame(expected, index=contingency.index, columns=contingency.columns).round(1))
print(f"\nχ² statistic : {chi2_stat:.4f}")
print(f"Degrees of freedom: {dof}  (= (r-1)(c-1) = {contingency.shape[0]-1} × {contingency.shape[1]-1})")
print(f"p-value      : {p_val:.4f}")
print()
if p_val < 0.05:
    print("p < 0.05 → reject H₀: student status and gender are NOT independent.")
else:
    print("p ≥ 0.05 → fail to reject H₀: no evidence of association between student status and gender.")

# Visualise observed vs expected
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
contingency.plot(kind='bar', ax=axes[0], rot=0, title='Observed counts')
pd.DataFrame(expected, index=contingency.index,
             columns=contingency.columns).plot(kind='bar', ax=axes[1], rot=0, title='Expected counts (under H₀)')
for ax in axes:
    ax.set_xlabel('Student'); ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# ── Visual comparison of the four key distributions ──────────────────────────
from scipy import stats as sp_stats

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Distributions behind the four inference concepts', fontsize=13, y=1.02)

# 1. t-distribution (t-test & CI)
ax = axes[0]
x = np.linspace(-4, 4, 300)
for df, ls, label in [(2, ':', 'df=2'), (5, '--', 'df=5'), (30, '-', 'df=30')]:
    ax.plot(x, sp_stats.t.pdf(x, df), ls=ls, lw=2, label=label)
ax.plot(x, sp_stats.norm.pdf(x), lw=1.5, color='grey', alpha=0.5, label='Normal')
t_crit = sp_stats.t.ppf(0.975, df=30)
ax.fill_between(x, sp_stats.t.pdf(x, 30), where=np.abs(x) >= t_crit,
                alpha=0.25, color='red', label='reject region (α=0.05)')
ax.set_title("t-distribution\n(t-test & CI)")
ax.legend(fontsize=7); ax.set_xlabel('t')

# 2. Normal / sampling distribution of β̂₁ (confidence interval)
ax = axes[1]
x = np.linspace(-3, 3, 300)
ax.plot(x, sp_stats.norm.pdf(x), lw=2, color='steelblue', label='N(0,1)')
ci_lo, ci_hi = sp_stats.norm.ppf(0.025), sp_stats.norm.ppf(0.975)
ax.fill_between(x, sp_stats.norm.pdf(x), where=(x >= ci_lo) & (x <= ci_hi),
                alpha=0.3, color='steelblue', label='95% CI region')
ax.axvline(ci_lo, color='red', ls='--', lw=1)
ax.axvline(ci_hi, color='red', ls='--', lw=1, label=f'±{ci_hi:.2f}')
ax.set_title("Sampling dist of β̂\n(Confidence Interval)")
ax.legend(fontsize=7); ax.set_xlabel('standardised β̂')

# 3. p-value illustration on t-dist
ax = axes[2]
x = np.linspace(-5, 5, 300)
pdf = sp_stats.t.pdf(x, df=30)
t_obs = 2.5
ax.plot(x, pdf, lw=2, color='steelblue')
ax.fill_between(x, pdf, where=np.abs(x) >= t_obs,
                alpha=0.4, color='red', label=f'p-value (t={t_obs})')
ax.axvline(t_obs, color='red', lw=2, ls='--')
ax.axvline(-t_obs, color='red', lw=2, ls='--')
p_shown = 2 * sp_stats.t.sf(t_obs, 30)
ax.set_title(f"p-value illustration\np = {p_shown:.3f}")
ax.legend(fontsize=7); ax.set_xlabel('t')

# 4. Chi-squared distribution
ax = axes[3]
x = np.linspace(0, 25, 300)
for df, ls, label in [(1, ':', 'df=1'), (3, '--', 'df=3'), (5, '-', 'df=5'), (10, '-.', 'df=10')]:
    ax.plot(x, sp_stats.chi2.pdf(x, df), ls=ls, lw=2, label=label)
chi2_crit = sp_stats.chi2.ppf(0.95, df=5)
ax.fill_between(x, sp_stats.chi2.pdf(x, 5), where=x >= chi2_crit,
                alpha=0.25, color='red', label=f'reject (df=5, α=0.05)')
ax.set_title("χ² distribution\n(Chi-squared test)")
ax.legend(fontsize=7); ax.set_xlabel('χ²'); ax.set_xlim(0, 25)

plt.tight_layout()
plt.show()

### Concept: Chi-Squared Test (χ²)

#### What it is
The **chi-squared test** is used when your outcome variable is **categorical** (counts, frequencies).  
It asks: *"Are two categorical variables independent, or is there a relationship between them?"*

Two main uses:
1. **Goodness-of-fit** — does the observed distribution match an expected one?
2. **Test of independence** — are two categorical variables associated (e.g., gender × student status)?

#### Analogy
You roll a die 600 times and count how often each face appears.  
Under H₀ (fair die), you'd expect 100 of each.  
The χ² statistic measures how far off your observed counts are from 100:

- If the die is fair → small χ², large p-value → can't reject H₀
- If the die is loaded → large χ², small p-value → reject H₀

More generally: χ² measures the total "surprise" across all categories.

#### Formula

$$\chi^2 = \sum_{\text{all cells}} \frac{(\text{Observed} - \text{Expected})^2}{\text{Expected}}$$

For a **test of independence** in an $r \times c$ contingency table:

$$\text{Expected}_{ij} = \frac{(\text{row } i \text{ total}) \times (\text{column } j \text{ total})}{n}$$

$$\chi^2 = \sum_{i=1}^r \sum_{j=1}^c \frac{(O_{ij} - E_{ij})^2}{E_{ij}}$$

#### Distribution
Under $H_0$, the statistic follows a **chi-squared distribution**:

$$\chi^2 \sim \chi^2_k$$

where the degrees of freedom $k$ depend on the test:
- **Goodness-of-fit:** $k = (\text{number of categories}) - 1$
- **Independence:** $k = (r-1)(c-1)$

The χ² distribution is **right-skewed** and only takes non-negative values.  
Large χ² → evidence against H₀ → we only look at the **right tail**.

#### Key differences from t-test

| Feature | t-test | χ²-test |
|---|---|---|
| Variable type | Continuous | Categorical (counts) |
| H₀ | β = 0 (no linear effect) | Variables are independent |
| Distribution | Symmetric t | Right-skewed χ² |
| Tail | Two-sided | One-sided (right) |

### Concept: P-Value

#### What it is
The **p-value** is the probability of observing a test statistic at least as extreme as the one we got, **assuming the null hypothesis is true**.

$$p = P\bigl(|T| \geq |t_{\text{obs}}| \mid H_0 \text{ true}\bigr)$$

A **small p-value** (< 0.05) means: "If there truly were no effect, results this extreme would be very unlikely — so we reject H₀."

> ⚠️ The p-value is **not** "the probability that H₀ is true."  
> It's not "the probability the result is due to chance."  
> It tells you about the **data given H₀**, not about H₀ given the data.

#### Analogy
You're a juror. The null hypothesis is "the defendant is innocent."

- A **small p-value** = the evidence (data) is very improbable under innocence → you lean toward rejecting innocence.
- A **large p-value** = the evidence is consistent with innocence → you don't have enough to convict.

The p-value does **not** tell you how guilty (or true) the alternative is — just how inconsistent the data is with H₀.

#### Visual interpretation

```
   ────────────────────────────────────────────
   0 ──────────────── 0.05 ──────────────────── 1
   │← reject H₀ →│         │← fail to reject H₀ →│
   "unlikely if H₀ true"     "consistent with H₀"
```

The p-value is the **shaded tail area** under the t-distribution beyond your observed |t|.

#### Common thresholds
| p-value | Conventional interpretation |
|---|---|
| < 0.001 | Very strong evidence against H₀ |
| 0.001 – 0.01 | Strong evidence |
| 0.01 – 0.05 | Moderate evidence (conventional cutoff) |
| 0.05 – 0.10 | Weak / suggestive |
| > 0.10 | Little to no evidence |

#### Formula (two-sided)
$$p = 2 \cdot P(T_{n-p-1} > |t_{\text{obs}}|) = 2 \cdot \bigl[1 - F_{t_{n-p-1}}(|t_{\text{obs}}|)\bigr]$$

where $F_{t_k}$ is the CDF of the t-distribution with $k$ degrees of freedom.

In [ ]:
beta1 = sm_slr.params['TV']
se_beta1 = sm_slr.bse['TV']
n = len(y)
dof = n - 2   # degrees of freedom for SLR

t_stat = beta1 / se_beta1
p_value = 2 * stats.t.sf(abs(t_stat), df=dof)   # two-sided

print(f"β̂₁      = {beta1:.6f}")
print(f"SE(β̂₁)  = {se_beta1:.6f}")
print(f"t        = β̂₁ / SE(β̂₁) = {t_stat:.4f}")
print(f"dof      = {dof}")
print(f"p-value  = 2 × P(T > |t|) = {p_value:.2e}")
print()
print("Conclusion: p << 0.05 → reject H₀: β₁ = 0 → TV spend IS associated with sales.")

In [ ]:
# Visualise the t-distribution and where our t-statistic falls
t_range = np.linspace(-20, 20, 1000)
pdf_vals = stats.t.pdf(t_range, df=dof)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(t_range, pdf_vals, lw=2, color='steelblue')
ax.axvline(t_stat, color='red', lw=2, label=f't = {t_stat:.1f}')
ax.axvline(-t_stat, color='red', lw=2, ls='--')
ax.fill_between(t_range, pdf_vals, where=np.abs(t_range) >= abs(t_stat),
                alpha=0.3, color='red', label=f'p-value = {p_value:.2e}')
ax.set_xlabel('t'); ax.set_ylabel('density')
ax.set_title(f't-distribution (df={dof}) — H₀: β₁ = 0')
ax.legend(); ax.set_xlim(-22, 22)
plt.tight_layout()
plt.show()

---
## 3. Model Accuracy: RSE and R²

In [ ]:
# --- via statsmodels ---
rss   = sm_slr.ssr           # residual sum of squares
tss   = sm_slr.centered_tss  # total sum of squares
r2_sm = sm_slr.rsquared
rse   = np.sqrt(sm_slr.mse_resid)   # RSE = sqrt(RSS / (n-2))

print("=== SLR (TV only) ===")
print(f"RSS   = {rss:.2f}")
print(f"TSS   = {tss:.2f}")
print(f"R²    = (TSS - RSS) / TSS = {r2_sm:.3f}")
print(f"RSE   = √(RSS / (n-2))    = {rse:.3f}")
print()

# --- via sklearn metrics ---
r2_sk  = r2_score(y, y_hat)
mse_sk = mean_squared_error(y, y_hat)
print("sklearn equivalents:")
print(f"  r2_score          = {r2_sk:.3f}")
print(f"  RMSE              = {np.sqrt(mse_sk):.3f}  (note: sklearn divides by n, not n-2)")
print()

# --- manual calculation ---
tss_manual = np.sum((y - y.mean())**2)
rss_manual = np.sum(residuals**2)
r2_manual  = (tss_manual - rss_manual) / tss_manual
rse_manual = np.sqrt(rss_manual / (n - 2))
print("Manual:")
print(f"  R²  = {r2_manual:.3f}")
print(f"  RSE = {rse_manual:.3f}")
print(f"\nInterpretation: TV explains {r2_manual*100:.1f}% of variance in sales.")
print(f"Typical prediction error: ±{rse_manual:.2f}k units  (mean sales = {y.mean():.1f}k → {rse_manual/y.mean()*100:.1f}% error)")

### 3.1 R² vs correlation

In SLR, $R^2 = r^2$ where $r$ is the Pearson correlation coefficient.

In [ ]:
r = np.corrcoef(x, y)[0, 1]
print(f"Pearson r      = {r:.4f}")
print(f"r²             = {r**2:.4f}")
print(f"R² from model  = {r2_manual:.4f}")
print("\nIn SLR: R² = r²  ✓")

---
## 4. Multiple Linear Regression

**Model:** $Y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \cdots + \beta_p X_p + \varepsilon$

Each $\beta_j$ = expected change in Y per unit increase in $X_j$, **holding all other predictors fixed**.

### 4.1 Correlation matrix — understanding multicollinearity

In [ ]:
corr = adv.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            vmin=-1, vmax=1, ax=ax, square=True)
ax.set_title('Advertising — correlation matrix')
plt.tight_layout()
plt.show()

print("\nCorr(radio, newspaper) =", corr.loc['radio','newspaper'].round(3))
print("This explains why newspaper appears significant alone but not in MLR.")

### 4.2 Suppressor effect — newspaper alone vs. newspaper + radio

This demonstrates the "woes of interpreting regression coefficients".

In [ ]:
# Newspaper alone
m_news = smf.ols('sales ~ newspaper', data=adv).fit()
print("=== Newspaper alone ===")
print(m_news.params.round(4), "  p:", m_news.pvalues.round(4))
print()

# Newspaper + radio
m_both = smf.ols('sales ~ radio + newspaper', data=adv).fit()
print("=== Radio + Newspaper ===")
print(m_both.params.round(4), "  p:", m_both.pvalues.round(4))
print()
print("Once radio is in the model, newspaper's coefficient is effectively ZERO.")
print("Reason: newspaper is correlated with radio (r=0.35); its apparent effect was radio's.")

### 4.3 Full MLR — TV + radio + newspaper

In [ ]:
X_all = adv[['TV', 'radio', 'newspaper']]
X_all_const = sm.add_constant(X_all)
sm_mlr = sm.OLS(y, X_all_const).fit()
print(sm_mlr.summary())

In [ ]:
# Compare SLR vs MLR key metrics
metrics = pd.DataFrame({
    'Model': ['SLR (TV only)', 'MLR (TV+radio+newspaper)'],
    'R²':    [sm_slr.rsquared, sm_mlr.rsquared],
    'Adj R²': [sm_slr.rsquared_adj, sm_mlr.rsquared_adj],
    'RSE':   [np.sqrt(sm_slr.mse_resid), np.sqrt(sm_mlr.mse_resid)],
    'F-stat': [sm_slr.fvalue, sm_mlr.fvalue],
})
metrics = metrics.set_index('Model').round(3)
print(metrics)

In [ ]:
# Coefficient bar chart with error bars
coefs = sm_mlr.params.drop('const')
cis   = sm_mlr.conf_int().drop('const')
errors = np.array([
    coefs.values - cis[0].values,
    cis[1].values - coefs.values
])

fig, ax = plt.subplots(figsize=(6, 3))
ax.barh(coefs.index, coefs.values, xerr=errors, color=['steelblue','orange','grey'],
        capsize=5, alpha=0.8)
ax.axvline(0, color='black', lw=1)
ax.set_xlabel('Coefficient estimate')
ax.set_title('MLR coefficients with 95 % CI')
plt.tight_layout()
plt.show()

---
## 5. Variable Selection

### 5.1 F-test — is at least one predictor useful?

$$F = \frac{(\text{TSS} - \text{RSS})/p}{\text{RSS}/(n-p-1)}$$

In [ ]:
p_vars = 3   # TV, radio, newspaper
F_stat = sm_mlr.fvalue
F_pval = sm_mlr.f_pvalue

print(f"F-statistic : {F_stat:.1f}")
print(f"p-value     : {F_pval:.2e}")
print()

# Manual F calculation
y_hat_mlr = sm_mlr.fittedvalues
rss_mlr = np.sum((y - y_hat_mlr)**2)
F_manual = ((tss - rss_mlr) / p_vars) / (rss_mlr / (n - p_vars - 1))
print(f"Manual F    : {F_manual:.1f}")

# Compare to F critical value at α = 0.05
F_crit = stats.f.ppf(0.95, dfn=p_vars, dfd=n-p_vars-1)
print(f"F_crit(0.05): {F_crit:.2f}")
print(f"\nF ({F_stat:.0f}) >> F_crit ({F_crit:.1f}) → reject H₀ — at least one predictor helps.")

### 5.2 Forward selection — greedy variable addition

In [ ]:
def forward_selection_aic(df, target):
    """Greedy forward selection by AIC using statsmodels."""
    remaining = set(df.columns) - {target}
    selected = []
    current_aic = smf.ols(f'{target} ~ 1', data=df).fit().aic

    print(f"Start  : AIC = {current_aic:.2f}, predictors = []")
    while remaining:
        aics = {}
        for var in remaining:
            formula = f"{target} ~ {' + '.join(selected + [var])}"
            aics[var] = smf.ols(formula, data=df).fit().aic
        best_var = min(aics, key=aics.get)
        if aics[best_var] < current_aic:
            selected.append(best_var)
            remaining.remove(best_var)
            current_aic = aics[best_var]
            print(f"Add {best_var:<12}: AIC = {current_aic:.2f}, predictors = {selected}")
        else:
            print(f"Stop   : adding any variable increases AIC.")
            break
    return selected

selected_vars = forward_selection_aic(adv, 'sales')

### 5.3 Backward elimination — greedy variable removal

In [ ]:
def backward_elimination(df, target, threshold_pval=0.05):
    """Remove predictors with p-value above threshold one at a time."""
    predictors = [c for c in df.columns if c != target]
    while predictors:
        formula = f"{target} ~ {' + '.join(predictors)}"
        result = smf.ols(formula, data=df).fit()
        pvals = result.pvalues.drop('Intercept', errors='ignore')
        max_pval = pvals.max()
        if max_pval > threshold_pval:
            drop_var = pvals.idxmax()
            print(f"Remove {drop_var:<12}: p = {max_pval:.4f}, remaining = {[p for p in predictors if p != drop_var]}")
            predictors.remove(drop_var)
        else:
            print(f"Stop   : all p-values < {threshold_pval}, final model: {predictors}")
            break
    return predictors

final_vars = backward_elimination(adv, 'sales')

### 5.4 Model selection criteria — AIC, BIC, Adjusted R²

In [ ]:
from itertools import combinations

features = ['TV', 'radio', 'newspaper']
rows = []

for k in range(1, len(features)+1):
    for combo in combinations(features, k):
        formula = f"sales ~ {' + '.join(combo)}"
        m = smf.ols(formula, data=adv).fit()
        rows.append({
            'predictors': list(combo),
            'k': k,
            'R²': m.rsquared,
            'Adj R²': m.rsquared_adj,
            'AIC': m.aic,
            'BIC': m.bic,
            'RSE': np.sqrt(m.mse_resid),
        })

sel_df = pd.DataFrame(rows)
print(sel_df.to_string(index=False))
print()
print("Best by AIC:", sel_df.loc[sel_df['AIC'].idxmin(), 'predictors'])
print("Best by BIC:", sel_df.loc[sel_df['BIC'].idxmin(), 'predictors'])
print("Best by Adj R²:", sel_df.loc[sel_df['Adj R²'].idxmax(), 'predictors'])

---
## 6. Qualitative Predictors

We use a simulated Credit dataset to replicate the ISLR examples.

In [ ]:
# Simulate Credit dataset (ISLR-like)
rng = np.random.default_rng(0)
N = 400
income   = rng.uniform(10, 180, N)
limit    = 1000 + 5*income + rng.normal(0, 500, N)
rating   = limit / 5 + rng.normal(0, 20, N)
gender   = rng.choice(['Male', 'Female'], N, p=[0.5, 0.5])
student  = rng.choice(['No', 'Yes'], N, p=[0.7, 0.3])
ethnicity = rng.choice(['African American', 'Asian', 'Caucasian'], N, p=[0.15, 0.30, 0.55])
balance  = (0.8*income + 0.05*limit
            + 19.73*(gender=='Female').astype(float)
            + 400*(student=='Yes').astype(float)
            - 18.69*(ethnicity=='Asian').astype(float)
            - 12.50*(ethnicity=='Caucasian').astype(float)
            + rng.normal(0, 200, N))
balance = np.clip(balance, 0, None)

credit = pd.DataFrame({
    'income': income, 'limit': limit, 'rating': rating,
    'balance': balance, 'gender': gender,
    'student': student, 'ethnicity': ethnicity,
})
print(credit.head())
print(credit.dtypes)

### 6.1 Binary dummy variable — gender

In [ ]:
# pandas creates a dummy automatically in the formula
m_gender = smf.ols('balance ~ C(gender)', data=credit).fit()
print(m_gender.summary().tables[1])
print()
print("Interpretation:")
print("  Intercept       = expected balance for MALE (baseline)")
print("  gender[T.Male]  = difference Male - Female")

# Alternative: manual dummy with pd.get_dummies
credit['female'] = (credit['gender'] == 'Female').astype(int)
m_gender2 = sm.OLS(credit['balance'], sm.add_constant(credit['female'])).fit()
print("\nManual dummy:")
print(m_gender2.params.round(2))

In [ ]:
# Visualise
fig, ax = plt.subplots(figsize=(5, 3))
for g, grp in credit.groupby('gender'):
    ax.hist(grp['balance'], bins=25, alpha=0.5, label=g)
ax.axvline(credit[credit.gender=='Male']['balance'].mean(), color='steelblue', lw=2, ls='--', label='Male mean')
ax.axvline(credit[credit.gender=='Female']['balance'].mean(), color='orange', lw=2, ls='--', label='Female mean')
ax.set_xlabel('Balance'); ax.legend()
ax.set_title('Balance by gender')
plt.tight_layout()
plt.show()

### 6.2 Multi-level factor — ethnicity (3 levels, 2 dummies)

In [ ]:
# African American is the baseline (alphabetically first, or set explicitly)
m_eth = smf.ols('balance ~ C(ethnicity, Treatment(reference="African American"))', data=credit).fit()
print(m_eth.summary().tables[1])
print()
print("β₀ = expected balance for African American (baseline)")
print("β_Asian = AA-Asian difference")
print("β_Caucasian = AA-Caucasian difference")

# Using pd.get_dummies — equivalent
eth_dummies = pd.get_dummies(credit['ethnicity'], drop_first=True).astype(int)
print("\nDummy columns:", eth_dummies.columns.tolist())
print(eth_dummies.head())

---
## 7. Interactions & Non-linearity

### 7.1 Interaction term: TV × radio

In [ ]:
# Additive model
m_add = smf.ols('sales ~ TV + radio', data=adv).fit()

# Interaction model
m_int = smf.ols('sales ~ TV + radio + TV:radio', data=adv).fit()

print("Additive model:")
print(m_add.summary().tables[1])
print(f"\n  R² = {m_add.rsquared:.4f}")

print("\n" + "="*60)
print("Interaction model:")
print(m_int.summary().tables[1])
print(f"\n  R² = {m_int.rsquared:.4f}")

In [ ]:
# Interpret the interaction coefficient
b1 = m_int.params['TV']
b3 = m_int.params['TV:radio']

print("Effective slope of TV depends on radio spend:")
for radio_level in [0, 20, 40]:
    eff_slope = b1 + b3 * radio_level
    print(f"  radio = {radio_level:3d} → $1k more TV → +{eff_slope*1000:.0f} units")

In [ ]:
# Visualise: interaction surface
tv_grid    = np.linspace(0, 300, 60)
radio_grid = np.linspace(0, 50, 50)
TV_g, R_g  = np.meshgrid(tv_grid, radio_grid)

beta = m_int.params
sales_pred = (beta['Intercept']
              + beta['TV'] * TV_g
              + beta['radio'] * R_g
              + beta['TV:radio'] * TV_g * R_g)

fig = plt.figure(figsize=(9, 5))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(TV_g, R_g, sales_pred, alpha=0.6, cmap='viridis')
ax.scatter(adv['TV'], adv['radio'], adv['sales'],
           c='red', s=10, alpha=0.4)
ax.set_xlabel('TV'); ax.set_ylabel('Radio'); ax.set_zlabel('Sales')
ax.set_title('Interaction model — curved surface')
plt.tight_layout()
plt.show()

### 7.2 Qualitative × quantitative interaction — student × income

In [ ]:
m_no_int  = smf.ols('balance ~ income + C(student)', data=credit).fit()
m_with_int = smf.ols('balance ~ income * C(student)', data=credit).fit()

income_range = np.linspace(credit['income'].min(), credit['income'].max(), 200)

def predict_line(model, income_vals, student_val):
    df = pd.DataFrame({'income': income_vals, 'student': student_val})
    return model.predict(df)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, model, title in zip(axes,
                             [m_no_int, m_with_int],
                             ['No interaction (parallel lines)', 'With interaction (different slopes)']):
    for student_val, color, label in [('Yes','tomato','student'), ('No','steelblue','non-student')]:
        preds = predict_line(model, income_range, student_val)
        ax.plot(income_range, preds, color=color, lw=2, label=label)
    ax.set_xlabel('Income'); ax.set_ylabel('Balance')
    ax.set_title(title); ax.legend()

plt.tight_layout()
plt.show()

### 7.3 Non-linearity: polynomial regression on Auto data

**mpg = β₀ + β₁·horsepower + β₂·horsepower² + ε**

In [ ]:
# Load Auto dataset
try:
    from sklearn.datasets import fetch_openml
    auto_raw = fetch_openml('autompg', version=1, as_frame=True, parser='auto')
    auto = auto_raw.frame[['horsepower', 'mpg']].dropna().copy()
    auto['horsepower'] = pd.to_numeric(auto['horsepower'], errors='coerce')
    auto = auto.dropna()
except Exception:
    # Fallback: simulate data consistent with ISLR Auto summary stats
    rng2 = np.random.default_rng(1)
    hp = rng2.uniform(46, 230, 392)
    mpg = 56.9 - 0.4662*hp + 0.0012*hp**2 + rng2.normal(0, 4, 392)
    auto = pd.DataFrame({'horsepower': hp, 'mpg': mpg})

hp = auto['horsepower'].values.reshape(-1, 1)
mpg = auto['mpg'].values

print(f"Auto dataset: {len(auto)} rows")
print(auto.describe())

In [ ]:
# Fit degree 1, 2, 5 using sklearn Pipeline
hp_range = np.linspace(hp.min(), hp.max(), 300).reshape(-1, 1)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(hp, mpg, alpha=0.3, edgecolors='k', linewidths=0.3, s=15, label='data')

colors = {'degree 1 (linear)': 'orange', 'degree 2 (quadratic)': 'steelblue', 'degree 5': 'green'}

for deg, label in [(1, 'degree 1 (linear)'), (2, 'degree 2 (quadratic)'), (5, 'degree 5')]:
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=deg, include_bias=False)),
        ('lr',   LinearRegression()),
    ])
    pipe.fit(hp, mpg)
    r2 = r2_score(mpg, pipe.predict(hp))
    ax.plot(hp_range, pipe.predict(hp_range), lw=2,
            color=colors[label], label=f'{label}  (R²={r2:.3f})')

ax.set_xlabel('Horsepower'); ax.set_ylabel('MPG')
ax.set_title('Polynomial regression on Auto data')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# statsmodels OLS for the degree-2 model — check the ISLR table numbers
auto['hp']  = auto['horsepower']
auto['hp2'] = auto['horsepower'] ** 2

m_poly2 = smf.ols('mpg ~ hp + hp2', data=auto).fit()
print(m_poly2.summary())

**Expected:** β̂(hp) ≈ −0.466, β̂(hp²) ≈ 0.0012, both with p < 0.0001 (matching ISLR slide 46).

In [ ]:
# Residuals: does quadratic fix the non-linearity?
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, deg, title in zip(axes, [1, 2], ['Linear fit residuals', 'Quadratic fit residuals']):
    pipe = Pipeline([('poly', PolynomialFeatures(degree=deg, include_bias=False)),
                     ('lr', LinearRegression())])
    pipe.fit(hp, mpg)
    resid = mpg - pipe.predict(hp)
    ax.scatter(pipe.predict(hp), resid, alpha=0.3, s=15, edgecolors='k', linewidths=0.3)
    ax.axhline(0, color='red', lw=1, ls='--')
    ax.set_xlabel('Fitted values'); ax.set_ylabel('Residuals')
    ax.set_title(title)

plt.tight_layout()
plt.show()

print("Linear residuals show clear pattern (curve) → model is misspecified.")
print("Quadratic residuals are much more random → better fit.")

---
## Exercises

Use the **California housing dataset** (from sklearn) for Exercises 1–3, and the **Credit dataset** (already loaded above) for Exercise 4.

```python
from sklearn.datasets import fetch_california_housing
cal_raw = fetch_california_housing(as_frame=True)
cal = cal_raw.frame  # 20 640 rows, 8 features + MedHouseVal target
```

In [ ]:
from sklearn.datasets import fetch_california_housing
cal_raw = fetch_california_housing(as_frame=True)
cal = cal_raw.frame
print(cal.shape)
print(cal.describe().round(2))

### Exercise 1 — Simple Linear Regression

Fit a SLR of `MedHouseVal` (median house value, $100k) on `MedInc` (median income) using **statsmodels**.

Tasks:
1. Report β̂₀ and β̂₁ with their 95 % confidence intervals.
2. What is the p-value for the slope? Is it statistically significant?
3. What are the RSE and R²?
4. Plot the scatter with the fitted line and residuals.

In [ ]:
# Your code here


#### Exercise 1 — Solution

In [ ]:
m_ex1 = smf.ols('MedHouseVal ~ MedInc', data=cal).fit()
print(m_ex1.summary())

print(f"\nRSE = {np.sqrt(m_ex1.mse_resid):.4f}")
print(f"R²  = {m_ex1.rsquared:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
inc_range = np.linspace(cal['MedInc'].min(), cal['MedInc'].max(), 300)

# Scatter + fit
sample = cal.sample(1000, random_state=0)  # plot subset for speed
axes[0].scatter(sample['MedInc'], sample['MedHouseVal'], alpha=0.2, s=10)
axes[0].plot(inc_range,
             m_ex1.params['Intercept'] + m_ex1.params['MedInc']*inc_range,
             color='red', lw=2)
axes[0].set_xlabel('MedInc'); axes[0].set_ylabel('MedHouseVal')
axes[0].set_title('SLR: MedHouseVal ~ MedInc')

# Residuals
resid_ex1 = m_ex1.resid
axes[1].scatter(m_ex1.fittedvalues, resid_ex1, alpha=0.1, s=5)
axes[1].axhline(0, color='red', lw=1)
axes[1].set_xlabel('Fitted'); axes[1].set_ylabel('Residuals')
axes[1].set_title('Residuals vs Fitted')

plt.tight_layout()
plt.show()

### Exercise 2 — Multiple Linear Regression

Extend the model to include `MedInc`, `HouseAge`, and `AveRooms`.

Tasks:
1. Fit the MLR and print the full summary.
2. Compare R² and RSE to the SLR from Exercise 1.
3. Which predictor has the largest t-statistic in absolute value?
4. Is the F-statistic significant?

In [ ]:
# Your code here


#### Exercise 2 — Solution

In [ ]:
m_ex2 = smf.ols('MedHouseVal ~ MedInc + HouseAge + AveRooms', data=cal).fit()
print(m_ex2.summary())

print(f"\nSLR  R² = {m_ex1.rsquared:.4f}  RSE = {np.sqrt(m_ex1.mse_resid):.4f}")
print(f"MLR  R² = {m_ex2.rsquared:.4f}  RSE = {np.sqrt(m_ex2.mse_resid):.4f}")

tvals = m_ex2.tvalues.drop('Intercept')
print(f"\nLargest |t|: {tvals.abs().idxmax()}  (t = {tvals.abs().max():.2f})")
print(f"F-statistic: {m_ex2.fvalue:.1f}  p = {m_ex2.f_pvalue:.2e}")

### Exercise 3 — Interaction term

Add the interaction `MedInc × HouseAge` to the MLR from Exercise 2.

Tasks:
1. Fit the model `MedHouseVal ~ MedInc + HouseAge + AveRooms + MedInc:HouseAge`.
2. Is the interaction term statistically significant?
3. By how much does R² change?
4. Interpret: for a house aged 20 years vs 40 years, how does the income slope change?

In [ ]:
# Your code here


#### Exercise 3 — Solution

In [ ]:
m_ex3 = smf.ols('MedHouseVal ~ MedInc + HouseAge + AveRooms + MedInc:HouseAge', data=cal).fit()
print(m_ex3.summary().tables[1])

print(f"\nR² without interaction: {m_ex2.rsquared:.4f}")
print(f"R² with    interaction: {m_ex3.rsquared:.4f}")
print(f"Δ R² = {m_ex3.rsquared - m_ex2.rsquared:.4f}")

b_inc = m_ex3.params['MedInc']
b_int = m_ex3.params['MedInc:HouseAge']
print()
for age in [20, 40]:
    eff = b_inc + b_int * age
    print(f"  HouseAge={age}: income slope = {eff:.4f}  ($100k per unit income)")

### Exercise 4 — Qualitative predictors

Using the **Credit dataset** (loaded in Section 6), fit a model predicting `balance` from `income`, `student`, and their interaction.

Tasks:
1. Fit the model with `income`, `student`, and `income:student` interaction.
2. Report the intercept and slope for each group (student vs non-student).
3. Plot the two fitted lines (income vs balance, coloured by student status).
4. Are students and non-students on parallel income–balance lines, or do they differ in slope?

In [ ]:
# Your code here


#### Exercise 4 — Solution

In [ ]:
m_ex4 = smf.ols('balance ~ income * C(student)', data=credit).fit()
print(m_ex4.summary().tables[1])

b0 = m_ex4.params['Intercept']
b_inc = m_ex4.params['income']
b_student = m_ex4.params.get('C(student)[T.Yes]', 0)
b_int = m_ex4.params.get('income:C(student)[T.Yes]', 0)

print(f"\nNon-student: balance = {b0:.1f} + {b_inc:.2f}·income")
print(f"Student:     balance = {b0+b_student:.1f} + {b_inc+b_int:.2f}·income")

inc_range = np.linspace(credit['income'].min(), credit['income'].max(), 200)
fig, ax = plt.subplots(figsize=(7, 4))
for s_val, s_label, color in [('No','non-student','steelblue'), ('Yes','student','tomato')]:
    sub = credit[credit.student == s_val]
    ax.scatter(sub['income'], sub['balance'], alpha=0.2, s=10, color=color)
    df_pred = pd.DataFrame({'income': inc_range, 'student': s_val})
    ax.plot(inc_range, m_ex4.predict(df_pred), lw=2, color=color, label=s_label)
ax.set_xlabel('Income'); ax.set_ylabel('Balance')
ax.set_title('Balance ~ Income × Student')
ax.legend()
plt.tight_layout()
plt.show()

pval_int = m_ex4.pvalues.get('income:C(student)[T.Yes]', None)
if pval_int is not None:
    print(f"\nInteraction p-value: {pval_int:.4f}")
    if pval_int < 0.05:
        print("Significant → students and non-students have DIFFERENT income slopes.")
    else:
        print("Not significant → lines are approximately parallel (different intercepts only).")